In [ ]:
import json
import google.generativeai as genai
from time import sleep
import re
import random
from collections import defaultdict

# === Cấu hình API Key Gemini ===
genai.configure(api_key="")

output_file = ""
log_file = ""
batch_size = 6
target_samples = 2000
temperature = 0.9  
max_retry = 3

model_priority_list = [
    'gemini-2.5-flash-lite',
    'gemini-2.5-flash', 
    'gemini-2.5-pro',
    'gemini-2.0-flash',
    'gemini-2.0-flash-lite',
]

max_output_tokens = 8192

class DiversityEnhancedGenerator:
    def __init__(self):
        self.generated_themes = set()
        self.theme_categories = {
            "rejection": [
                "nấu ăn", "lập trình", "sức khỏe", "thể thao", "tài chính", 
                "du lịch", "thời trang", "làm đẹp", "xe cộ", "nhà đất",
                "giáo dục", "công việc", "mối quan hệ", "sở thích", "thú cưng",
                "phim ảnh", "âm nhạc", "sách", "công nghệ", "thời tiết"
            ],
            "question_types": [
                "hướng dẫn", "tư vấn", "so sánh", "kinh nghiệm", "đánh giá",
                "khuyến nghị", "giải thích", "phân tích", "dự đoán", "gợi ý"
            ],
            "question_formats": [
                "Làm thế nào để {action}?",
                "Bạn có thể tư vấn về {topic} không?",
                "Nên chọn {option_a} hay {option_b}?",
                "Kinh nghiệm về {topic} như thế nào?",
                "Cách tốt nhất để {action} là gì?",
                "Bạn biết gì về {topic}?",
                "Tôi muốn học về {topic}",
                "Có nên {action} không?",
                "Địa chỉ nào tốt cho {service}?",
                "Sản phẩm nào tốt cho {purpose}?"
            ]
        }
        
        self.rejection_responses = [
            "Xin lỗi, tôi chỉ có thể trả lời các câu hỏi về lịch sử Việt Nam. Câu hỏi về {topic} nằm ngoài phạm vi chuyên môn của tôi.",
            "Tôi xin lỗi, tôi chỉ tập trung vào lĩnh vực lịch sử Việt Nam. Tôi không thể hỗ trợ bạn với câu hỏi về {topic}.",
            "Tôi chỉ trả lời các câu hỏi liên quan đến lịch sử Việt Nam. Đối với thông tin về {topic}, bạn nên tham khảo các nguồn chuyên về lĩnh vực này.",
            "Xin lỗi, tôi không có chuyên môn về {topic}. Tôi chỉ có thể hỗ trợ các câu hỏi về lịch sử Việt Nam.",
            "Tôi chỉ chuyên về lịch sử Việt Nam. Câu hỏi về {topic} nằm ngoài khả năng của tôi."
        ]

    def get_diverse_prompt(self, batch_size):
        # Chọn ngẫu nhiên các chủ đề
        selected_themes = random.sample(self.theme_categories["rejection"], 
                                      min(8, len(self.theme_categories["rejection"])))
        selected_types = random.sample(self.theme_categories["question_types"], 3)
        
        prompt = f"""
HÃY TẠO {batch_size} MẪU HỘI THOẠI JSON CHO CHATBOT LỊCH SỬ VIỆT NAM VỚI ĐỘ ĐA DẠNG CAO:

YÊU CẦU QUAN TRỌNG VỀ ĐA DẠNG:
- MỖI mẫu phải có CHỦ ĐỀ KHÁC NHAU: {', '.join(selected_themes)}
- MỖI mẫu phải có DẠNG CÂU HỎI KHÁC NHAU: {', '.join(selected_types)}
- ĐẢM BẢO KHÔNG CÓ MẪU NÀO GIỐNG NHAU QUÁ 30%

HƯỚNG DẪN TẠO CÂU HỎI:
1. Sử dụng các cách diễn đạt đa dạng:
   - Câu hỏi trực tiếp: "Cách làm X?"
   - Câu nhờ tư vấn: "Bạn có thể tư vấn về Y?"  
   - Câu hỏi so sánh: "Giữa A và B cái nào tốt hơn?"
   - Câu hỏi kinh nghiệm: "Kinh nghiệm về Z?"
   - Câu yêu cầu: "Tôi muốn biết về W"

2. Đa dạng độ dài và cấu trúc:
   - Câu ngắn: "Cách nấu phở?"
   - Câu dài có ngữ cảnh: "Tôi đang muốn học nấu các món ăn Hà Nội, bạn có thể hướng dẫn không?"
   - Câu có nhiều thông tin: "Tôi cần tư vấn về đầu tư chứng khoán cho người mới bắt đầu"

3. Sử dụng từ ngữ tự nhiên, đa dạng

HƯỚNG DẪN TẠO CÂU TRẢ LỜI:
- Đa dạng cách từ chối, không dùng quá 2 lần cùng một công thức
- Giữ tone lịch sự, chuyên nghiệp
- Giải thích rõ giới hạn chuyên môn

VÍ DỤ ĐA DẠNG:
- "Làm thế nào để nấu phở bò ngon?" → "Xin lỗi, tôi chỉ chuyên về lịch sử..."
- "Bạn có thể tư vấn cho tôi về đầu tư chứng khoán?" → "Tôi xin lỗi, lĩnh vực tài chính..."
- "Kinh nghiệm du lịch Đà Lạt như thế nào?" → "Tôi chỉ trả lời các câu hỏi về lịch sử..."

ĐỊNH DẠNG JSON:
[
  {{
    "messages": [
      {{"role": "system", "content": "Bạn là chuyên gia lịch sử Việt Nam."}},
      {{"role": "user", "content": "QUESTION"}},
      {{"role": "assistant", "content": "ANSWER"}}
    ]
  }}
]

QUAN TRỌNG: CHỈ TRẢ VỀ JSON, KHÔNG THÊM VĂN BẢN NÀO KHÁC.
"""

        return prompt

def get_available_model():
    for model_name in model_priority_list:
        try:
            model = genai.GenerativeModel(model_name)
            print(f"Sử dụng model: {model_name}")
            return model_name
        except Exception as e:
            print(f"Model {model_name} không khả dụng: {e}")
            continue
    return model_priority_list[0]

def clean_json_response(text):
    cleaned = re.sub(r'```json\s*', '', text)
    cleaned = re.sub(r'\s*```', '', cleaned)
    cleaned = cleaned.strip()
    cleaned = re.sub(r'^[^{[]*', '', cleaned)
    cleaned = re.sub(r'[^}\]]*$', '', cleaned)
    return cleaned

def repair_truncated_json(json_str):
    if not json_str.strip():
        return json_str
        
    open_braces = json_str.count('{')
    close_braces = json_str.count('}')
    open_brackets = json_str.count('[')
    close_brackets = json_str.count(']')
    
    repaired = json_str
    
    if open_braces > close_braces:
        repaired += '}' * (open_braces - close_braces)
    if open_brackets > close_brackets:
        repaired += ']' * (open_brackets - close_brackets)
    
    return repaired

def extract_and_parse_json(response_text):
    print(f"🔧 Đang xử lý response dài {len(response_text)} chars...")
    
    cleaned_text = clean_json_response(response_text)
    
    try:
        json_data = json.loads(cleaned_text)
        print(" Parse trực tiếp thành công")
        return json_data
    except json.JSONDecodeError:
        print(" Parse trực tiếp thất bại")
    
    array_pattern = r'\[\s*\{[\s\S]*?\}\s*\]'
    array_matches = re.findall(array_pattern, cleaned_text, re.DOTALL)
    
    if array_matches:
        json_str = max(array_matches, key=len)
        json_str_repaired = repair_truncated_json(json_str)
        
        try:
            json_data = json.loads(json_str_repaired)
            print(" Parse JSON đã sửa thành công")
            return json_data
        except json.JSONDecodeError:
            print(" Vẫn lỗi JSON sau khi sửa")
    
    start_idx = cleaned_text.find('[')
    end_idx = cleaned_text.rfind(']')
    
    if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
        json_str = cleaned_text[start_idx:end_idx+1]
        json_str_repaired = repair_truncated_json(json_str)
        
        try:
            json_data = json.loads(json_str_repaired)
            print(" Parse JSON manual thành công")
            return json_data
        except json.JSONDecodeError:
            print(" Lỗi parse JSON manual")
    
    raise ValueError("Không thể trích xuất JSON từ response")

def generate_rejection_batch(batch_size):
    generator = DiversityEnhancedGenerator()
    prompt = generator.get_diverse_prompt(batch_size)

    current_model = get_available_model()
    
    for attempt in range(1, max_retry + 1):
        try:
            print(f" Attempt {attempt} với model {current_model}...")
            
            model = genai.GenerativeModel(current_model)
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=temperature,
                    max_output_tokens=max_output_tokens,
                    top_p=0.9
                )
            )
            
            response_text = response.text.strip()
            print(f" Raw response length: {len(response_text)} chars")
            
            json_data = extract_and_parse_json(response_text)
            
            if not isinstance(json_data, list):
                raise ValueError("Kết quả không phải là list")
                
            print(f" Đã tạo được {len(json_data)} samples")
                
            for i, item in enumerate(json_data):
                if "messages" not in item:
                    raise ValueError(f"Thiếu key 'messages' trong item {i}")
                    
            return json_data
            
        except Exception as e:
            print(f" Lỗi attempt {attempt}: {str(e)[:200]}")
            
            if attempt < max_retry:
                next_model_index = (model_priority_list.index(current_model) + 1) % len(model_priority_list)
                current_model = model_priority_list[next_model_index]
                print(f" Chuyển sang model: {current_model}")
                sleep(3)
            else:
                return []

def create_fallback_rejection_batch(batch_size):
    fallback_data = []
    generator = DiversityEnhancedGenerator()
    
    for i in range(batch_size):
        theme = random.choice(generator.theme_categories["rejection"])
        question_format = random.choice(generator.theme_categories["question_formats"])
        response_template = random.choice(generator.rejection_responses)
        
        if "{action}" in question_format:
            question = question_format.format(action=f"thực hiện {theme}")
        elif "{topic}" in question_format:
            question = question_format.format(topic=theme)
        elif "{option_a}" in question_format:
            question = question_format.format(option_a=f"phương án A cho {theme}", option_b=f"phương án B cho {theme}")
        else:
            question = question_format.format(service=theme, purpose=theme)
            
        answer = response_template.format(topic=theme)
        
        messages = [
            {"role": "system", "content": "Bạn là chuyên gia lịch sử Việt Nam."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ]
        fallback_data.append({"messages": messages})
    
    return fallback_data

# === XỬ LÝ CHÍNH ===
processed_count = 0
diversity_generator = DiversityEnhancedGenerator()

print(f" Bắt đầu tạo {target_samples} samples data từ chối với độ đa dạng cao...")

with open(output_file, "w", encoding="utf-8") as f_out, \
     open(log_file, "w", encoding="utf-8") as f_log:

    while processed_count < target_samples:
        current_batch_size = min(batch_size, target_samples - processed_count)
        print(f"\n Đang tạo batch {processed_count + 1} đến {processed_count + current_batch_size}...")
        
        batch_data = generate_rejection_batch(current_batch_size)
        
        if not batch_data:
            print(" Sử dụng fallback...")
            batch_data = create_fallback_rejection_batch(current_batch_size)
        
        for data in batch_data:
            f_out.write(json.dumps(data, ensure_ascii=False) + "\n")
            processed_count += 1
        
        f_log.write(f"Đã tạo {processed_count}/{target_samples} samples\n")
        print(f" Đã tạo {processed_count}/{target_samples} samples")
        
        if processed_count < target_samples:
            sleep(5)

print(f"\n Hoàn tất! Đã tạo {processed_count} samples data từ chối với độ đa dạng cao.")